In [ ]:
!pip install torch torchvision timm scikit-learn albumentations optuna

import os, glob, random
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import timm


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 14.5 MB/s eta 0:00:00


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive',force_remount=True)

Mounted at /content/drive


In [ ]:
# Path to dataset
data_dir = "/content/drive/MyDrive/SAM_enhanced"
labels_df = pd.read_csv("/content/drive/MyDrive/data.csv")

# Clean missing
labels_df = labels_df.dropna(subset=["superclass", "subclass", "image_id"]).reset_index(drop=True)

# Folder name = superclass_subclass
labels_df["folder"] = labels_df["superclass"] + "_" + labels_df["subclass"]

# Encode labels
le = LabelEncoder()
labels_df["label"] = le.fit_transform(labels_df["subclass"])
print("Classes:", le.classes_)

# Split dataset
train_df, temp_df = train_test_split(labels_df, test_size=0.3, stratify=labels_df["label"], random_state=42)
val_df, test_df   = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")


Classes: ['bd' 'md' 'pd']
Train: 378, Val: 81, Test: 81


In [ ]:
def preprocess_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.GaussianBlur(img, (5,5), 0)
    img = cv2.resize(img, (224,224))
    return img

augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Affine(scale=(0.9,1.1), translate_percent=(0.1,0.1),
             rotate=(-15,15), shear=(-5,5), p=0.5),
    A.RandomBrightnessContrast(p=0.5),
    A.GaussianBlur(blur_limit=(3,5), p=0.3),
    A.Normalize(mean=(0.485,0.456,0.406),
                std=(0.229,0.224,0.225)),
    ToTensorV2()
])


In [ ]:
def tile_image_for_mil(img_rgb, patch_size=224, stride=224, max_patches=64, mode="grid"):
    H, W = img_rgb.shape[:2]
    patches = []
    for y in range(0, max(1, H - patch_size + 1), stride):
        for x in range(0, max(1, W - patch_size + 1), stride):
            crop = img_rgb[y:y+patch_size, x:x+patch_size]
            if crop.shape[:2] == (patch_size, patch_size):
                patches.append(crop)
    if len(patches) > max_patches:
        idx = np.linspace(0, len(patches)-1, max_patches).astype(int)
        patches = [patches[i] for i in idx]
    if len(patches) == 0:
        patches = [cv2.resize(img_rgb, (patch_size, patch_size))]
    return patches

class MILBagDataset(Dataset):
    def __init__(self, img_root, df, augment, patch_size=224, stride=224, max_patches=32):
        self.img_root, self.df, self.augment = img_root, df.reset_index(drop=True), augment
        self.patch_size, self.stride, self.max_patches = patch_size, stride, max_patches

    def _resolve_img_path(self, row):

        folder = row["folder"]
        image_id = str(row["image_id"])

    # Multiple possible patterns
        patterns = [
        f"{folder}_20x_{image_id}_clahe*",
        f"{folder}_40x_{image_id}_clahe*",
        f"{folder}_20x_{image_id}*",
        f"{folder}_40x_{image_id}*",
        f"{folder}_{image_id}*",
        f"{image_id}*"
    ]

        for patt in patterns:
          files = glob.glob(os.path.join(self.img_root, folder, patt))
          if files:
            return files[0]

    # If nothing found, return None instead of crashing
        return None

    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
      row = self.df.iloc[idx]
      img_path = self._resolve_img_path(row)
      if img_path is None:
        raise FileNotFoundError(f"No file found for row: {row.to_dict()}")

      raw = cv2.imread(img_path)
      raw = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
      raw = cv2.GaussianBlur(raw, (5,5), 0)

      patches = tile_image_for_mil(raw, patch_size=self.patch_size, stride=self.stride,
                                 max_patches=self.max_patches)
      tensor_patches = [self.augment(image=p)["image"] for p in patches]
      bag_tensor = torch.stack(tensor_patches, dim=0)
      return bag_tensor, int(row["label"]), row["image_id"]


def mil_collate(batch):
    bags, labels, ids = zip(*batch)
    return list(bags), torch.tensor(labels, dtype=torch.long), list(ids)


In [ ]:
class MILModel(nn.Module):
    def __init__(self, backbone, num_classes, agg="mean"):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.num_features, num_classes)
        self.agg = agg

    def forward(self, bag):
        if bag.dim() == 4:
            bag = bag.unsqueeze(0)
        B, n, C, H, W = bag.shape
        bag = bag.view(B*n, C, H, W)

        feats = self.backbone(bag)
        logits = self.classifier(feats)
        logits = logits.view(B, n, -1)

        if self.agg == "mean":
            bag_logits = logits.mean(dim=1)
        elif self.agg == "max":
            bag_logits = logits.max(dim=1).values
        elif self.agg == "lse":
            bag_logits = torch.logsumexp(logits, dim=1)
        else:
            bag_logits = logits.mean(dim=1)
        return bag_logits, logits


In [ ]:
criterion = nn.CrossEntropyLoss()

def run_epoch_mil(model, loader, optimizer=None, train=True, pooling="mean"):
    if train: model.train()
    else: model.eval()

    total_loss, all_preds, all_labels = 0.0, [], []
    for bags, labels, ids in loader:
        bag = bags[0].to(device)
        labels = labels.to(device)

        if train:
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=(device.type=="cuda")):
                inst_logits = model(bag)[0]  # bag_logits
                loss = criterion(inst_logits, labels)
            loss.backward()
            optimizer.step()
        else:
            with torch.no_grad():
                inst_logits = model(bag)[0]
                loss = criterion(inst_logits, labels)

        total_loss += float(loss.item())
        preds = inst_logits.argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / max(1, len(loader))
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="weighted", zero_division=0)
    return avg_loss, acc, prec, rec, f1


In [ ]:
import optuna
def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-4, 1e-2, log=True)
    bag_size = trial.suggest_categorical("bag_size", [8, 16, 32])
    pooling = trial.suggest_categorical("pooling", ["mean", "max", "lse"])
    epochs = 5

    # datasets
    train_dataset = MILBagDataset(data_dir, train_df, augment, max_patches=bag_size)
    val_dataset   = MILBagDataset(data_dir, val_df,   augment, max_patches=bag_size)

    # filter missing files (skip rows with no file)
    train_dataset.df = train_dataset.df[train_dataset.df.apply(lambda r: train_dataset._resolve_img_path(r) is not None, axis=1)]
    val_dataset.df   = val_dataset.df[val_dataset.df.apply(lambda r: val_dataset._resolve_img_path(r) is not None, axis=1)]

    train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=mil_collate)
    val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False, collate_fn=mil_collate)

    # model
    backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
    model = MILModel(backbone, num_classes=len(le.classes_), agg=pooling).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val_acc = 0
    for _ in range(epochs):
        run_epoch_mil(model, train_loader, optimizer=optimizer, train=True, pooling=pooling)
        vl_loss, vl_acc, _, _, _ = run_epoch_mil(model, val_loader, train=False, pooling=pooling)
        best_val_acc = max(best_val_acc, vl_acc)

    return best_val_acc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=10)  # try 10 sets

print("Best hyperparameters:", study.best_params)
print("Best Val Acc:", study.best_value)


[I 2026-02-05 10:46:22,623] A new study created in memory with name: no-name-d6a65e6b-e297-4e52-a2f4-2a919e562dc5
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

[I 2026-02-05 10:56:53,789] Trial 0 finished with value: 0.3950617283950617 and parameters: {'lr': 0.00015698029126787097, 'weight_decay': 0.0011509515849085987, 'bag_size': 8, 'pooling': 'mean'}. Best is trial 0 with value: 0.3950617283950617.
[I 2026-02-05 11:03:36,637] Trial 1 finished with value: 0.37037037037037035 and parameters: {'lr': 0.0003694485619740192, 'weight_decay': 0.00010946289532337294, 'bag_size': 16, 'pooling': 'max'}. Best is trial 0 with value: 0.3950617283950617.
[I 2026-02-05 11:10:19,993] Trial 2 finished with value: 0.37037037037037035 and parameters: {'lr': 0.0002671205303846748, 'weight_decay': 0.002531714266127874, 'bag_size': 16, 'pooling': 'lse'}. Best is trial 0 with value: 0.3950617283950617.
[I 2026-02-05 11:15:33,274] Trial 3 finished with value: 0.5925925925925926 and parameters: {'lr': 1.711975724063776e-05, 'weight_decay': 0.0003731813917295257, 'bag_size': 8, 'pooling': 'max'}. Best is trial 3 with value: 0.5925925925925926.
[I 2026-02-05 11:22:14

Best hyperparameters: {'lr': 1.711975724063776e-05, 'weight_decay': 0.0003731813917295257, 'bag_size': 8, 'pooling': 'max'}
Best Val Acc: 0.5925925925925926


In [ ]:
best_params = study.best_params

train_dataset = MILBagDataset(data_dir, train_df, augment, max_patches=best_params["bag_size"])
val_dataset   = MILBagDataset(data_dir, val_df,   augment, max_patches=best_params["bag_size"])
test_dataset  = MILBagDataset(data_dir, test_df,  augment, max_patches=best_params["bag_size"])

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True,  collate_fn=mil_collate)
val_loader   = DataLoader(val_dataset,   batch_size=1, shuffle=False, collate_fn=mil_collate)
test_loader  = DataLoader(test_dataset,  batch_size=1, shuffle=False, collate_fn=mil_collate)

backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=0)
model = MILModel(backbone, num_classes=len(le.classes_), agg=best_params["pooling"]).to(device)

optimizer = optim.AdamW(model.parameters(),
                        lr=best_params["lr"],
                        weight_decay=best_params["weight_decay"])

best_val_acc, stale, patience = 0.0, 0, 5
EPOCHS = 20

for epoch in range(1, EPOCHS+1):
    tr_loss, tr_acc, tr_prec, tr_rec, tr_f1 = run_epoch_mil(model, train_loader, optimizer, train=True, pooling=best_params["pooling"])
    vl_loss, vl_acc, vl_prec, vl_rec, vl_f1 = run_epoch_mil(model, val_loader, train=False, pooling=best_params["pooling"])

    print(f"Epoch {epoch:02d}/{EPOCHS} | Train Acc {tr_acc:.4f} | Val Acc {vl_acc:.4f} | F1 {vl_f1:.4f}")

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        stale = 0
        torch.save(model.state_dict(), "swin_tiny_best_tuned.pt")
    else:
        stale += 1
        if stale >= patience:
            print("⏹️ Early stopping.")
            break

print("✅ Best Val Acc after tuning:", best_val_acc)


Epoch 01/20 | Train Acc 0.3439 | Val Acc 0.5432 | F1 0.4480
Epoch 02/20 | Train Acc 0.4021 | Val Acc 0.4074 | F1 0.3461
Epoch 03/20 | Train Acc 0.4286 | Val Acc 0.2840 | F1 0.1256
Epoch 04/20 | Train Acc 0.4603 | Val Acc 0.6420 | F1 0.6418
Epoch 05/20 | Train Acc 0.5423 | Val Acc 0.5926 | F1 0.5467
Epoch 06/20 | Train Acc 0.5529 | Val Acc 0.5679 | F1 0.5565
Epoch 07/20 | Train Acc 0.6429 | Val Acc 0.5062 | F1 0.4367
Epoch 08/20 | Train Acc 0.6587 | Val Acc 0.5926 | F1 0.5861
Epoch 09/20 | Train Acc 0.6958 | Val Acc 0.6667 | F1 0.6554
Epoch 10/20 | Train Acc 0.7328 | Val Acc 0.7037 | F1 0.7037
Epoch 11/20 | Train Acc 0.7831 | Val Acc 0.6914 | F1 0.6830
Epoch 12/20 | Train Acc 0.7751 | Val Acc 0.7654 | F1 0.7654
Epoch 13/20 | Train Acc 0.8280 | Val Acc 0.7407 | F1 0.7399
Epoch 14/20 | Train Acc 0.8492 | Val Acc 0.7284 | F1 0.7369
Epoch 15/20 | Train Acc 0.8651 | Val Acc 0.7160 | F1 0.7250
Epoch 16/20 | Train Acc 0.8757 | Val Acc 0.8272 | F1 0.8280
Epoch 17/20 | Train Acc 0.8704 | Val Acc

In [ ]:
import os, glob
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Mounted at /content/drive
Using device: cuda


In [ ]:
labels_df = pd.read_csv("/content/drive/MyDrive/data.csv")
labels_df = labels_df.dropna(subset=["superclass", "subclass", "image_id"]).reset_index(drop=True)
labels_df["folder"] = labels_df["superclass"] + "_" + labels_df["subclass"]

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
labels_df["label"] = le.fit_transform(labels_df["subclass"])
num_classes = len(le.classes_)
print("Classes:", le.classes_)


Classes: ['bd' 'md' 'pd']


In [ ]:
from sklearn.model_selection import train_test_split

_, temp_df = train_test_split(
    labels_df,
    test_size=0.3,
    stratify=labels_df["label"],
    random_state=42
)

_, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df["label"],
    random_state=42
)

print("Test samples:", len(test_df))


Test samples: 81


In [ ]:
eval_transform = A.Compose([
    A.Resize(224,224),
    A.Normalize(mean=(0.485,0.456,0.406),
                std=(0.229,0.224,0.225)),
    ToTensorV2()
])


In [ ]:
def tile_image_for_mil(img_rgb, patch_size=224, stride=224, max_patches=32):
    H, W = img_rgb.shape[:2]
    patches = []
    for y in range(0, max(1, H - patch_size + 1), stride):
        for x in range(0, max(1, W - patch_size + 1), stride):
            crop = img_rgb[y:y+patch_size, x:x+patch_size]
            if crop.shape[:2] == (patch_size, patch_size):
                patches.append(crop)
    if len(patches) > max_patches:
        idx = np.linspace(0, len(patches)-1, max_patches).astype(int)
        patches = [patches[i] for i in idx]
    if len(patches) == 0:
        patches = [cv2.resize(img_rgb, (patch_size, patch_size))]
    return patches


In [ ]:
class MILBagDataset(Dataset):
    def __init__(self, img_root, df, transform, max_patches=32):
        self.img_root = img_root
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.max_patches = max_patches

    def _resolve_img_path(self, row):
        folder = row["folder"]
        image_id = str(row["image_id"])

        patterns = [
            f"{folder}_20x_{image_id}*",
            f"{folder}_40x_{image_id}*",
            f"{folder}_{image_id}*",
            f"{image_id}*"
        ]

        for p in patterns:
            files = glob.glob(os.path.join(self.img_root, folder, p))
            if files:
                return files[0]
        return None

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = self._resolve_img_path(row)
        if img_path is None:
            raise FileNotFoundError(row.to_dict())

        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        patches = tile_image_for_mil(img, max_patches=self.max_patches)
        bag = torch.stack([self.transform(image=p)["image"] for p in patches])

        return bag, int(row["label"]), row["image_id"], img_path


In [ ]:
class MILModel(nn.Module):
    def __init__(self, backbone, num_classes, agg="mean"):
        super().__init__()
        self.backbone = backbone
        self.classifier = nn.Linear(backbone.num_features, num_classes)
        self.agg = agg

    def forward(self, bag):
        if bag.dim() == 4:
            bag = bag.unsqueeze(0)
        B, N, C, H, W = bag.shape
        bag = bag.view(B*N, C, H, W)

        feats = self.backbone(bag)
        logits = self.classifier(feats)
        logits = logits.view(B, N, -1)

        if self.agg == "mean":
            return logits.mean(1)
        elif self.agg == "max":
            return logits.max(1).values
        else:
            return torch.logsumexp(logits, dim=1)


In [ ]:
backbone = timm.create_model("swin_tiny_patch4_window7_224", pretrained=False, num_classes=0)
model = MILModel(backbone, num_classes=num_classes, agg="mean")
model.load_state_dict(torch.load("/content/swin_tiny_best_tuned.pt", map_location=device))
model.to(device)
model.eval()


MILModel(
  (backbone): SwinTransformer(
    (patch_embed): PatchEmbed(
      (proj): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
      (norm): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
    )
    (layers): Sequential(
      (0): SwinTransformerStage(
        (downsample): Identity()
        (blocks): Sequential(
          (0): SwinTransformerBlock(
            (norm1): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (attn): WindowAttention(
              (qkv): Linear(in_features=96, out_features=288, bias=True)
              (attn_drop): Dropout(p=0.0, inplace=False)
              (proj): Linear(in_features=96, out_features=96, bias=True)
              (proj_drop): Dropout(p=0.0, inplace=False)
              (softmax): Softmax(dim=-1)
            )
            (drop_path1): Identity()
            (norm2): LayerNorm((96,), eps=1e-05, elementwise_affine=True)
            (mlp): Mlp(
              (fc1): Linear(in_features=96, out_features=384, bias=True)

In [ ]:
def evaluate_false_negatives(data_root, name):
    dataset = MILBagDataset(data_root, test_df, eval_transform)
    loader = DataLoader(dataset, batch_size=1, shuffle=False)

    y_true, y_pred = [], []
    records = []

    with torch.no_grad():
        for bag, label, img_id, path in loader:
            bag = bag.to(device)
            out = model(bag)
            pred = out.argmax(1).item()

            y_true.append(label.item())
            y_pred.append(pred)

            records.append({
                "image_id": img_id[0],
                "true_label": le.classes_[label.item()],
                "pred_label": le.classes_[pred],
                "path": path[0]
            })

    cm = confusion_matrix(y_true, y_pred)
    print(f"\n📊 {name} Confusion Matrix\n", cm)

    fn_per_class = {}
    for i, cls in enumerate(le.classes_):
        fn_per_class[cls] = cm[i].sum() - cm[i,i]

    print(f"\n❌ False Negatives ({name})")
    for k,v in fn_per_class.items():
        print(f"{k}: {v}")

    df = pd.DataFrame(records)
    fn_df = df[df.true_label != df.pred_label]
    fn_df.to_csv(f"/content/FN_{name}.csv", index=False)

    return fn_df


In [ ]:
# Preprocessed dataset
fn_pre = evaluate_false_negatives(
    "/content/drive/MyDrive/SAM_enhanced",
    "Preprocessed"
)

# Raw dataset
fn_raw = evaluate_false_negatives(
    "/content/drive/MyDrive/LungHist700/data/images",
    "Raw"
)



📊 Preprocessed Confusion Matrix
 [[ 7 17  6]
 [ 0 18  6]
 [ 0  5 22]]

❌ False Negatives (Preprocessed)
bd: 23
md: 6
pd: 5

📊 Raw Confusion Matrix
 [[29  1  0]
 [13  7  4]
 [15  3  9]]

❌ False Negatives (Raw)
bd: 1
md: 17
pd: 18


In [ ]:
# Preprocessed dataset
fn_pre = evaluate_false_negatives(
    "/content/drive/MyDrive/SAM_enhanced",
    "Preprocessed"
)

# Raw dataset
fn_raw = evaluate_false_negatives(
    "/content/drive/MyDrive/LungHist700_clahe",
    "Raw"
)


📊 Preprocessed Confusion Matrix
 [[ 7 17  6]
 [ 0 18  6]
 [ 0  5 22]]

❌ False Negatives (Preprocessed)
bd: 23
md: 6
pd: 5

📊 Raw Confusion Matrix
 [[14  9  7]
 [ 0 15  9]
 [ 0  2 25]]

❌ False Negatives (Raw)
bd: 16
md: 9
pd: 2
